In [5]:
%load_ext autoreload
%autoreload 2

import sys
import os
import json

sys.path.append(os.path.abspath('../src'))
from utils.data_loader import WeatherCalibrator

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
with open('../src/config/ports.json', 'r') as f:
    ports_dict = json.load(f)
with open('../src/config/schedule.json', 'r') as f:
    schedule_list = json.load(f)

In [8]:
path_x = '../data/aegean_tau_x.nc'
path_y = '../data/aegean_tau_y.nc'

calibrator = WeatherCalibrator(path_x, path_y, ports_dict, schedule_list)

print("Extracting, routing around landmasses, and normalizing...")
df_normalized = calibrator.process_and_normalize()

print("\n--- August Calibration (Harsh Summer Meltemi) ---")
aug_params, aug_k = calibrator.calibrate_month(df_normalized, target_month=8)
print(f"JACOBI_PARAMS = {aug_params}")
print("\nK_FACTORS (Includes all ports and transit midpoints):")
for node, k in aug_k.items():
    print(f"  '{node}': {k:.3f}")

Loading NetCDF datasets...
Extracting, routing around landmasses, and normalizing...

--- August Calibration (Harsh Summer Meltemi) ---
JACOBI_PARAMS = {'mu': 0.09712019562721252, 'theta': 0.018171296296296297, 'sigma': 0.0390932556445527}

K_FACTORS (Includes all ports and transit midpoints):
  'Rafina': 0.259
  'Andros': 0.963
  'Tinos': 1.038
  'Mykonos': 1.013
  'Paros': 0.952
  'Naxos': 0.921
  'Transit_Rafina_Andros': 0.483
  'Transit_Andros_Tinos': 1.042
  'Transit_Tinos_Mykonos': 1.024
  'Transit_Mykonos_Paros': 0.947
  'Transit_Paros_Naxos': 0.947
  'Transit_Naxos_Paros': 0.947
  'Transit_Paros_Mykonos': 0.947
  'Transit_Mykonos_Tinos': 1.024
  'Transit_Tinos_Andros': 1.042
  'Transit_Andros_Rafina': 0.483


In [9]:
print("\n--- Generating 12-Month Climatology Database ---")
annual_climatology = {}

# Loop through all 12 months
for month in range(1, 13):
    try:
        params, k_factors = calibrator.calibrate_month(df_normalized, target_month=month)
        
        # Convert np.float32 to standard Python floats for JSON serialization
        clean_params = {k: float(v) for k, v in params.items()}
        clean_k = {k: float(v) for k, v in k_factors.items()}
        
        annual_climatology[str(month)] = {
            "jacobi": clean_params,
            "k_factors": clean_k
        }
        print(f"Successfully calibrated Month {month:02d}")
    except Exception as e:
        print(f"Failed to calibrate Month {month}: {e}")

# Save the master configuration file
output_path = '../src/config/weather_params.json'
with open(output_path, 'w') as f:
    json.dump(annual_climatology, f, indent=4)

print(f"\nSuccess! 12-month climate data saved to: {output_path}")


--- Generating 12-Month Climatology Database ---
Successfully calibrated Month 01
Successfully calibrated Month 02
Successfully calibrated Month 03
Successfully calibrated Month 04
Successfully calibrated Month 05
Successfully calibrated Month 06
Successfully calibrated Month 07
Successfully calibrated Month 08
Successfully calibrated Month 09
Successfully calibrated Month 10
Successfully calibrated Month 11
Successfully calibrated Month 12

Success! 12-month climate data saved to: ../src/config/weather_params.json
